# Truth-Serum Debate: BTS as a Judge for 2-Debater Disagreement

Two LLM debaters answer a binary factual question and predict the *other* debater's answer. A Bayesian Truth Serum (BTS) score (Prelec, Science 2004) rewards "surprisingly common" answers -- the property that incentivises truthful reporting -- and we use it to pick a winner without ever telling the judge the ground truth. We then check, post-hoc, whether the BTS-winner correlates with correctness on a small set of common-knowledge true/false items.

See `THEORY.md` for the derivation and assumptions. This notebook runs a *cheap* demo (N=20 questions, K=1 replicate, Haiku debater + Sonnet judge); estimated cost is under USD 2 at current pricing. Reviewers should clear outputs and re-run.

In [ ]:
import os, sys, json
from pathlib import Path

# Make repo root importable when notebook lives in notebooks/.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env or env"

from anthropic import Anthropic
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.debate import DebateConfig
from src.eval import run_eval

client = Anthropic()
cfg = DebateConfig(
    debater_model=os.environ.get("DEBATER_MODEL", "claude-haiku-4-5"),
    judge_model=os.environ.get("JUDGE_MODEL", "claude-sonnet-4-5"),
)
print("debater:", cfg.debater_model, "  judge:", cfg.judge_model)

In [ ]:
# Run mini-eval. ~20 LLM round-trips per debater + 0 for judge (BTS is closed-form).
import time
from src.eval import run_eval
run_eval._t0 = time.time()
df = run_eval(
    questions_path=REPO_ROOT / "data" / "questions.jsonl",
    out_dir=REPO_ROOT / "results",
    n=20,
    k=1,
    cfg=cfg,
    client=client,
    verbose=True,
)
df.head()

In [ ]:
# Plot: per-question BTS gap (winner_bts - loser_bts) vs. winner correctness.
df["bts_gap"] = (df[["bts_a", "bts_b"]].max(axis=1)
                  - df[["bts_a", "bts_b"]].min(axis=1))
df["winner_bts"] = df[["bts_a", "bts_b"]].max(axis=1)

# Bin by BTS gap into ~5 quantile bins; report empirical correctness with Wilson CIs.
from math import sqrt
def wilson(k, n, z=1.96):
    if n == 0: return (0.0, 0.0, 0.0)
    p = k / n
    denom = 1 + z*z/n
    centre = (p + z*z/(2*n)) / denom
    half = z*sqrt(p*(1-p)/n + z*z/(4*n*n)) / denom
    return p, max(0.0, centre-half), min(1.0, centre+half)

df_sorted = df.sort_values("bts_gap").reset_index(drop=True)
n_bins = min(5, max(2, len(df_sorted) // 4))
df_sorted["bin"] = pd.qcut(df_sorted["bts_gap"], q=n_bins, duplicates="drop")
agg = df_sorted.groupby("bin", observed=True).agg(
    n=("winner_correct", "size"),
    k=("winner_correct", "sum"),
    gap_mid=("bts_gap", "mean"),
).reset_index(drop=True)
ci = agg.apply(lambda r: wilson(int(r.k), int(r.n)), axis=1, result_type="expand")
ci.columns = ["p", "lo", "hi"]
agg = pd.concat([agg, ci], axis=1)

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(
    agg["gap_mid"], agg["p"],
    yerr=[agg["p"] - agg["lo"], agg["hi"] - agg["p"]],
    fmt="o", capsize=4, lw=1.5, ms=8,
)
ax.axhline(0.5, color="grey", ls="--", lw=1, label="chance")
majority = df["majority_correct"].dropna().mean() if df["majority_correct"].notna().any() else None
if majority is not None:
    ax.axhline(majority, color="orange", ls=":", lw=1,
               label=f"majority-vote acc ({majority:.2f})")
ax.set_xlabel("BTS gap (winner score - loser score)")
ax.set_ylabel("P(BTS-winner == ground truth)")
ax.set_title("Does a larger BTS gap signal a more reliable winner?")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right")
fig.tight_layout()
out_png = REPO_ROOT / "results" / "bts_gap_vs_correctness.png"
fig.savefig(out_png, dpi=120)
print(f"Saved {out_png}")
plt.show()

## Interpretation

**What we measured.** For each of 20 binary common-knowledge items, two Haiku debaters answered independently and predicted the other's answer. We scored each debater with Prelec (2004) BTS and called the higher-BTS debater the winner. The plot bins questions by BTS gap and reports the winner's empirical correctness (Wilson 95% CIs).

**What to expect, honestly.** With N=20 the per-bin CIs span roughly +-25 points, so a flat curve is consistent with both "BTS gap predicts correctness" and "BTS is no better than coin-flip on this set". A flat curve is the most likely *null result* here, because (i) for easy common-knowledge items both debaters tend to agree, collapsing the BTS gap toward zero, and (ii) the BTS information term is degenerate for a 2-respondent population -- proper power needs more debaters.

**Next steps with more compute.** (a) Increase to >=200 questions across difficulty tiers (BoolQ-hard, TruthfulQA, MMLU-physics). (b) Increase to k>=3 debaters per round so the BTS info term has real degrees of freedom. (c) Replace the closed-form judge with a Sonnet adjudicator and compare. (d) Replicate with k=5 reps per question to separate sampling noise from method signal. (e) Run the Prelec (2004) lab-experiment-style calibration check to confirm BTS truly tracks truthfulness in this LLM setting -- it has only been validated on humans.